In [1]:
import pandas as pd

analysis_df = pd.read_csv('../data/analysis_df.csv')

print(f"Shape: {analysis_df.shape}")
print(f"\nColumns ({len(analysis_df.columns)}):")
print(analysis_df.columns.tolist())
print(f"\nMissing values:")
print(analysis_df.isna().sum()[analysis_df.isna().sum() > 0])
print(f"\nReview score distribution:")
print(analysis_df['review_score'].value_counts().sort_index())

Shape: (95809, 28)

Columns (28):
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'unique_sellers', 'unique_products', 'total_price', 'total_freight', 'first_seller_id', 'first_product_id', 'review_score', 'review_creation_date', 'review_answer_timestamp', 'product_id', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_photos_qty', 'seller_id', 'seller_state', 'product_category_name_english']

Missing values:
product_category_name            1349
product_weight_g                   16
product_length_cm                  16
product_height_cm                  16
product_width_cm                   16
product_photos_qty               1349
product_category_name_english    1368
dtype: int64

Review score distribution:
review_score
1     9380
2     2929
3     7919
4    18893


In [2]:
baseline_1star_rate = (analysis_df['review_score'] == 1).mean()
print(f"Baseline 1 star rate: {baseline_1star_rate}")

Baseline 1 star rate: 0.09790311974866661


## 03 — Exploration

### Baseline
1-star rate across analytical cohort: 9.8%

### Hypothesis 1: Delivery Performance
Longer delivery time and delays predict 1-star reviews.

In [3]:
# Convert datetime columns (CSV strips dtype)
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    analysis_df[col] = pd.to_datetime(analysis_df[col])

# Compute delivery metrics
analysis_df['actual_delivery_days'] = (
    analysis_df['order_delivered_customer_date'] - analysis_df['order_purchase_timestamp']
).dt.days

analysis_df['delivery_delay_days'] = (
    analysis_df['order_delivered_customer_date'] - analysis_df['order_estimated_delivery_date']
).dt.days

analysis_df['was_late'] = analysis_df['delivery_delay_days'] > 0

# Summary stats
print("Actual delivery days:")
print(analysis_df['actual_delivery_days'].describe())
print(f"\nDelivery delay days:")
print(analysis_df['delivery_delay_days'].describe())
print(f"\nLate deliveries: {analysis_df['was_late'].sum():,} ({analysis_df['was_late'].mean()*100:.1f}%)")

Actual delivery days:
count    95809.000000
mean        12.051759
std          9.465858
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        208.000000
Name: actual_delivery_days, dtype: float64

Delivery delay days:
count    95809.000000
mean       -11.911762
std         10.111347
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

Late deliveries: 6,380 (6.7%)


In [4]:
# Bucket delivery delay into meaningful groups
def delay_bucket(d):
    if pd.isna(d):
        return 'unknown'
    elif d < -7:
        return 'very early (>1wk early)'
    elif d < 0:
        return 'early (0-7 days early)'
    elif d == 0:
        return 'on time'
    elif d <= 7:
        return 'late (1-7 days late)'
    else:
        return 'very late (>1wk late)'

analysis_df['delay_bucket'] = analysis_df['delivery_delay_days'].apply(delay_bucket)

# 1-star rate by bucket
delay_analysis = analysis_df.groupby('delay_bucket').agg(
    n_orders=('order_id', 'count'),
    star_1_count=('review_score', lambda x: (x == 1).sum()),
    star_1_rate=('review_score', lambda x: (x == 1).mean())
).reset_index()

# Sort meaningfully
bucket_order = ['very early (>1wk early)', 'early (0-7 days early)', 'on time', 
                'late (1-7 days late)', 'very late (>1wk late)']
delay_analysis['delay_bucket'] = pd.Categorical(delay_analysis['delay_bucket'], 
                                                  categories=bucket_order, ordered=True)
delay_analysis = delay_analysis.sort_values('delay_bucket')

print(delay_analysis)
print(f"\nBaseline 1-star rate: 9.8%")

              delay_bucket  n_orders  star_1_count  star_1_rate
3  very early (>1wk early)     70915          4612     0.065036
0   early (0-7 days early)     17234          1225     0.071080
2                  on time      1280           110     0.085938
1     late (1-7 days late)      3599          1490     0.414004
4    very late (>1wk late)      2781          1943     0.698670

Baseline 1-star rate: 9.8%


Late delivery is the strongest factor we've identified. The 1-star rate jumps from ~8% for on-time-or-early orders to 41% for orders 1-7 days late, and to 70% for orders more than a week late. Even a single day of delay appears to dramatically shift customer sentiment.

## Finding 1: Delivery Delay

The 1-star rate scales dramatically with delivery delay:
- On-time or early: 6.5-8.6% (near or below 9.8% baseline)
- 1-7 days late: 41.4% (4.2x baseline)
- More than 1 week late: 69.9% (7.1x baseline)

The relationship is monotonic and steep. The discontinuity at the 
"on time" threshold suggests customers anchor on promised dates.

**Caveats to address in analysis:**
- Late delivery may be confounded with product category (heavier/fragile items)
- Estimated delivery dates may be calibrated unevenly across categories
- Effect should be tested while controlling for product and seller factors

In [5]:
seller_count_analysis = analysis_df.groupby('unique_sellers').agg(
    n_orders=('order_id', 'count'),
    star_1_rate=('review_score', lambda x: (x == 1).mean())
).reset_index()

print(seller_count_analysis)

   unique_sellers  n_orders  star_1_rate
0               1     94548     0.094407
1               2      1202     0.351913
2               3        54     0.481481
3               4         3     1.000000
4               5         2     1.000000


## Finding 2: Multi-Seller Orders

Orders with multiple sellers show dramatically elevated 1-star rates:
- 1 seller (94,548 orders): 9.4% 1-star — baseline
- 2 sellers (1,202 orders): 35.2% 1-star — 3.6x baseline
- 3+ sellers (59 orders): 50% 1-star — but small sample

The effect is real and meaningful — multi-seller orders are about 1.3% 
of total volume but produce a disproportionate share of dissatisfaction.

**Caveats:**
- Effect may overlap with delivery delay — multi-seller orders likely 
  have more complex logistics and arrive later
- Need stratified analysis: do multi-seller orders still show elevated 
  1-star rates when controlling for delivery on-time-ness?

In [6]:
analysis_df.groupby('unique_sellers').agg(
    n=('order_id', 'count'),
    star_1_rate=('review_score', lambda x: (x == 1).mean()),
    late_rate=('was_late', 'mean'),
    avg_delivery_days=('actual_delivery_days', 'mean')
).round(3)

,n,star_1_rate,late_rate,avg_delivery_days
unique_sellers,,,,
1,94548,0.094,0.067,12.097
2,1202,0.352,0.010,8.684
3,54,0.481,0.000,7.593
4,3,1.000,0.000,9.333
5,2,1.000,0.000,13.000


## Finding 2: Multi-Seller Orders (REVISED — confounding ruled out)

Multi-seller orders show 3.6x higher 1-star rates than single-seller orders.

CRITICAL: This effect is NOT explained by delivery performance. Multi-seller 
orders actually arrive faster and more on-time than single-seller orders 
(8.7 days avg, 1.0% late vs 12.1 days avg, 6.7% late for single-seller).

This confirms multi-seller status as an INDEPENDENT driver of customer 
dissatisfaction, separate from delivery experience.

Hypothesized mechanisms (to test in Step 7):
- Fragmented package arrival creates uncertainty
- Inconsistent quality standards across sellers
- Customer confusion about order structure

In [7]:
# 1-star rate by product category — top 10 worst and top 10 best
category_analysis = (
    analysis_df
    .dropna(subset=['product_category_name_english'])
    .groupby('product_category_name_english')
    .agg(
        n_orders=('order_id', 'count'),
        star_1_rate=('review_score', lambda x: (x == 1).mean())
    )
    .query('n_orders >= 100')  # only categories with enough sample
    .sort_values('star_1_rate', ascending=False)
)

print("Top 10 worst categories (highest 1-star rate):")
print(category_analysis.head(10))
print("\nTop 10 best categories (lowest 1-star rate):")
print(category_analysis.tail(10))

Top 10 worst categories (highest 1-star rate):
                               n_orders  star_1_rate
product_category_name_english                       
fashion_male_clothing               105     0.200000
office_furniture                   1236     0.172330
audio                               341     0.164223
fixed_telephony                     209     0.138756
home_confort                        368     0.138587
construction_tools_safety           152     0.138158
bed_bath_table                     9071     0.118510
furniture_decor                    6163     0.114068
computers_accessories              6469     0.113155
air_conditioning                    241     0.107884

Top 10 best categories (lowest 1-star rate):
                               n_orders  star_1_rate
product_category_name_english                       
food                                431     0.078886
pet_shop                           1674     0.078256
fashion_bags_accessories           1802     0.076582
fashio

## Finding 3: Product Category

Product category influences 1-star rates, with a 17-point spread between 
the worst (fashion_male_clothing, 20%) and best (food_drink, 2.8%) categories.

The pattern is more interpretable than the rate itself:
- Worst: assembly-required, aesthetic-judgment, technical-spec categories
- Best: consumables, low-stakes utility items

This suggests product complexity — not the product itself — is the driver. 
The effect is weaker than delivery delay (17pp range vs 63pp range) but 
real and independent.

**Caveats:**
- Some categories have small samples (<200 orders) and rates are noisy
- Category may correlate with seller quality — interior designers/furniture 
  sellers may differ from book retailers in service standards
- Need to test category effect while controlling for delivery and seller factors

In [8]:
analysis_df[['total_freight','total_price']].head()

,total_freight,total_price
0,8.72,29.99
1,22.76,118.70
2,19.22,159.90
3,27.20,45.00
4,8.72,19.90


In [9]:
analysis_df['total_price'].describe()

count    95809.000000
mean       136.802336
std        207.800922
min          0.850000
25%         45.900000
50%         86.250000
75%        149.900000
max      13440.000000
Name: total_price, dtype: float64

In [10]:
# Freight cost ratio — total_freight / total_price. Does high relative shipping cost predict 1-star reviews?

def freight_cost_bucket(d):
    if pd.isna(d):
        return 'Unknown'
    elif d < 0.25:
        return 'Low'
    elif d < 0.5:
        return 'Average'
    elif d < 0.75:
        return 'Above Average'
    elif d == 1:
        return 'High'
    else:
        return 'Extremely High'

analysis_df['freight_cost_ratio'] = analysis_df['total_freight'] / analysis_df['total_price']
analysis_df['freight_cost_bucket'] = analysis_df['freight_cost_ratio'].apply(freight_cost_bucket)

# 1-star rate by bucket
freight_cost_ratio_analysis = analysis_df.groupby('freight_cost_bucket').agg(
    n_orders=('order_id', 'count'),
    star_1_count=('review_score', lambda x: (x == 1).sum()),
    star_1_rate=('review_score', lambda x: (x == 1).mean())
).reset_index()

# Sort meaningfully
bucket_order = ['Low', 'Average', 'Above Average', 
                'High', 'Extremely High']
freight_cost_ratio_analysis['freight_cost_bucket'] = pd.Categorical(freight_cost_ratio_analysis['freight_cost_bucket'], 
                                                  categories=bucket_order, ordered=True)
freight_cost_ratio_analysis = freight_cost_ratio_analysis.sort_values('freight_cost_bucket')

print(freight_cost_ratio_analysis)
print(f"\nBaseline 1-star rate: 9.8%")


  freight_cost_bucket  n_orders  star_1_count  star_1_rate
3                 Low     53037          5048     0.095179
1             Average     27590          2780     0.100761
0       Above Average      8801           879     0.099875
2      Extremely High      6381           673     0.105469

Baseline 1-star rate: 9.8%


## Non-Finding: Freight Cost Ratio
Tested whether freight-to-price ratio predicts 1-star rates. No meaningful 
relationship across four buckets (range: 9.5%-10.5%, all within baseline 
noise of 9.8%). Customers appear indifferent to shipping cost as a fraction 
of order value.

In [11]:

def price_bucket(d):
    if pd.isna(d):
        return 'unknown'
    elif d < 50:
        return 'cheap'
    elif d < 100:
        return 'averagely priced'
    elif d < 150:
        return 'above average'
    else:
        return 'expensive'

analysis_df['price_bucket'] = analysis_df['total_price'].apply(price_bucket)

# 1-star rate by bucket
price_analysis = analysis_df.groupby('price_bucket').agg(
    n_orders=('order_id', 'count'),
    star_1_count=('review_score', lambda x: (x == 1).sum()),
    star_1_rate=('review_score', lambda x: (x == 1).mean())
).reset_index()

# Sort meaningfully
bucket_order = ['cheap', 'averagely priced', 'above average', 
                'expensive']
price_analysis['price_bucket'] = pd.Categorical(price_analysis['price_bucket'], 
                                                  categories=bucket_order, ordered=True)
price_analysis = price_analysis.sort_values('price_bucket')

print(price_analysis)
print(f"\nBaseline 1-star rate: 9.8%")


       price_bucket  n_orders  star_1_count  star_1_rate
2             cheap     28618          2334     0.081557
1  averagely priced     27548          2575     0.093473
0     above average     16250          1608     0.098954
3         expensive     23393          2863     0.122387

Baseline 1-star rate: 9.8%


## Finding 5: Order Value (provisional)
Higher-value orders show modestly elevated 1-star rates (8.2% → 12.2% from 
cheapest to most expensive quartile). Effect is monotonic but small (4pp range).

CAVEAT: This may be entirely explained by category — expensive orders likely 
concentrate in furniture and electronics categories we already identified 
as worse-performing. Need to test whether price effect persists within categories.

In [12]:
analysis_df['product_photos_qty'].describe()

count    94460.000000
mean         2.252054
std          1.748097
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         20.000000
Name: product_photos_qty, dtype: float64

In [13]:
def product_photos_qty_bucket(d):
    if pd.isna(d):
        return 'missing_data'
    elif d == 1:
        return 'Single Photo (1)'
    elif d <= 3:
        return 'Standard (2-3)'
    elif d <= 5:
        return 'High (4-5)'
    else:
        return 'Extreme (6+)'

analysis_df['product_photos_qty_bucket'] = analysis_df['product_photos_qty'].apply(product_photos_qty_bucket)

product_photos_qty_analysis = analysis_df.groupby('product_photos_qty_bucket').agg(
    n_orders=('order_id', 'count'),
    star_1_count=('review_score', lambda x: (x == 1).sum()),
    star_1_rate=('review_score', lambda x: (x == 1).mean())
).reset_index()

# Include all 5 categories in order, with missing_data last
bucket_order = ['Single Photo (1)', 'Standard (2-3)', 'High (4-5)', 'Extreme (6+)', 'missing_data']
product_photos_qty_analysis['product_photos_qty_bucket'] = pd.Categorical(
    product_photos_qty_analysis['product_photos_qty_bucket'], 
    categories=bucket_order, 
    ordered=True
)
product_photos_qty_analysis = product_photos_qty_analysis.sort_values('product_photos_qty_bucket')

# Sanity check
assert product_photos_qty_analysis['n_orders'].sum() == len(analysis_df), "Row count mismatch — buckets lost data"

print(product_photos_qty_analysis)

  product_photos_qty_bucket  n_orders  star_1_count  star_1_rate
2          Single Photo (1)     46645          4845     0.103870
3            Standard (2-3)     29508          2727     0.092416
1                High (4-5)     12194          1076     0.088240
0              Extreme (6+)      6113           548     0.089645
4              missing_data      1349           184     0.136397


## Finding 6: Product Photos & Data Completeness

(6a) Photo count weakly correlates with review quality. Range from 10.4% 
(1 photo) to 8.9% (6+ photos). Real but small effect.

(6b) ORDERS WITH MISSING PRODUCT DATA show 13.6% 1-star rate — 39% above baseline. 
This is not a product-feature finding but a meta-finding: products that lack 
metadata in Olist's catalog may be discontinued, removed, or low-quality. 
This warrants its own investigation in Step 7.

In [14]:
# Verification snippet
for bucket in analysis_df['product_photos_qty_bucket'].unique():
    if bucket == 'missing_data':
        continue
    values = analysis_df.loc[
        analysis_df['product_photos_qty_bucket'] == bucket, 
        'product_photos_qty'
    ].unique()
    print(f"{bucket}: contains values {sorted(values)}")

High (4-5): contains values [np.float64(4.0), np.float64(5.0)]
Single Photo (1): contains values [np.float64(1.0)]
Standard (2-3): contains values [np.float64(2.0), np.float64(3.0)]
Extreme (6+): contains values [np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0)]


In [15]:
import json
import os

# Make sure the dashboard data folder exists
os.makedirs('../dashboard/static/data', exist_ok=True)

# ---- Multi-seller data ----
multi_seller_data = (
    analysis_df.groupby('unique_sellers')
    .agg(
        n_orders=('order_id', 'count'),
        star_1_rate=('review_score', lambda x: (x == 1).mean()),
        late_rate=('was_late', 'mean'),
        avg_delivery_days=('actual_delivery_days', 'mean')
    )
    .reset_index()
)
# Only export buckets with meaningful sample size
multi_seller_data = multi_seller_data[multi_seller_data['n_orders'] >= 50]
multi_seller_data.to_json('../dashboard/static/data/multi_seller.json', orient='records')

# ---- Delivery delay data ----
delay_data = (
    analysis_df.groupby('delay_bucket')
    .agg(
        n_orders=('order_id', 'count'),
        star_1_rate=('review_score', lambda x: (x == 1).mean())
    )
    .reset_index()
)
delay_data.to_json('../dashboard/static/data/delivery_delay.json', orient='records')

# ---- Summary stats ----
summary = {
    'total_orders': int(len(analysis_df)),
    'baseline_1star_rate': float((analysis_df['review_score'] == 1).mean()),
    'multi_seller_or': 5.40,
    'delivery_or': 16.35,
    'category_cramers_v': 0.062
}
with open('../dashboard/static/data/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Data exported to dashboard/static/data/")

Data exported to dashboard/static/data/
